# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dwaynemongaya/flyrank-ml-internship_dwaynemongaya/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
from datasets import load_dataset
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    token=HF_TOKEN,
)

next(iter(ds["train"]))

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

{'report_date': datetime.date(2025, 1, 27),
 'client_hash_id': 'client_9958f0a7ae1df715',
 'content_hash_id': 'content_3b70a18ea133b2bb',
 'client_has_gsc': True,
 'client_has_ga4': True,
 'gsc_data_available': True,
 'ga4_data_available': False,
 'gsc_impressions': 30,
 'gsc_clicks': 0,
 'gsc_sum_position': 115,
 'gsc_avg_position': 3.8333333333333335,
 'ga4_pageviews': 0,
 'ga4_sessions': 0,
 'ga4_users': 0,
 'ga4_engaged_sessions': 0,
 'ga4_total_engagement_sec': 0,
 'sessions_organic': 0,
 'sessions_direct': 0,
 'sessions_referral': 0,
 'sessions_social': 0,
 'sessions_paid': 0,
 'sessions_ai': 0,
 'ai_chatgpt': 0,
 'ai_perplexity': 0,
 'ai_gemini': 0,
 'ai_copilot': 0,
 'ai_claude': 0,
 'ai_meta': 0,
 'ai_other': 0,
 'scroll_events': 0}

## 1. Unit of analysis + time window

The unit of analysis is one content page for one pseudonymized client on one report date. Each row represents the daily search and analytics performance of a single content item.

For this project, I will analyze a mid-panel month (for example, March 2026) instead of the final month. This allows me to build and evaluate features without using the natural outcome window, helping avoid information leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_engaged_sessions
- report_date

These are observable signals that can help rank pages for content review.

### Label / Proxy
- is_declining_label (proxy)

This proxy indicates whether a page is currently classified as declining.

### Context
- client_hash_id
- content_hash_id

These identify the client and content item but are not used as predictive features.

### Excluded
- trend_direction
- trend_pct

These are excluded because they are derived from the outcome and would introduce feature leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### DuckDB connection:

In [4]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

### Query 1: Verify the grain:

In [5]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │
│    date     │         varchar         │         varchar          │
├─────────────┼─────────────────────────┼──────────────────────────┤
│ 2025-01-27  │ client_9958f0a7ae1df715 │ content_3b70a18ea133b2bb │
│ 2025-01-27  │ client_9958f0a7ae1df715 │ content_fe8e8155ce1d47a2 │
│ 2025-01-27  │ client_9958f0a7ae1df715 │ content_b4462a1b90640058 │
│ 2025-01-27  │ client_9958f0a7ae1df715 │ content_c899aef92518c714 │
│ 2025-01-27  │ client_9958f0a7ae1df715 │ content_c7c1d2e68d9d0964 │
└─────────────┴─────────────────────────┴──────────────────────────┘

### Query 2: Row count + date span

In [6]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE date_trunc('month', report_date) = DATE '2026-03-01'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬────────────┐
│ row_count │ first_day  │  last_day  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

### Query 3: Availability (IS TRUE)

In [7]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE
    gsc_data_available IS TRUE
    AND date_trunc('month', report_date) = DATE '2026-03-01'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

## 3. Verify it with queries

### Query 1 – Grain

The first query confirms that each row represents one content page for one pseudonymized client on one report date.

### Query 2 – Row count and date span

Using March 2026 as the analysis month, the warehouse contains **9,841,378** rows ranging from **2026-03-01** to **2026-03-31**.

### Query 3 – Availability

Filtering with `gsc_data_available IS TRUE` leaves **3,611,061** usable rows. This verifies that the analysis is based only on observations where Google Search Console data is available.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This dataset cannot prove that one search or content factor directly causes better search performance. The warehouse contains an unbalanced history because different clients started contributing data at different times. Some early observations include only Google Search Console data, and overlapping time windows require careful feature design to avoid information leakage. Therefore, the results should be interpreted as decision-support rather than causal proof.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.